<a href="https://colab.research.google.com/github/Skulfre/Machine-Learning/blob/main/PyTorch_Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
train_data = datasets.FashionMNIST(root="data",
                             download=True,
                             target_transform=None,
                             train=True,
                             transform=ToTensor())
test_data = datasets.FashionMNIST(root='data',
                                  download=True,
                                  target_transform=None,
                                  train=False,
                                  transform=ToTensor())

In [ ]:
image, label = train_data[0]
image.shape, label

In [ ]:
len(train_data.data), len(train_data.targets), len(test_data.data), len(test_data.targets)

In [ ]:
class_names = train_data.classes
class_names

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 7))
plt.imshow(image.squeeze(), cmap='gray')
plt.title(class_names[label])

In [ ]:
t_image = image.T
plt.figure(figsize=(12, 7))
plt.imshow(t_image, cmap='gray')
plt.title(class_names[label])

In [ ]:
torch.manual_seed(42)
rows, cols = 4, 4
plt.figure(figsize=(12, 7))
for i in range(1, rows * cols + 1):
  rand_idx = torch.randint(0, len(train_data), size=[1,]).item()
  image, label = train_data[rand_idx]
  plt.subplot(rows, cols, i)
  plt.imshow(image.squeeze(), cmap='gray')
  plt.title(class_names[label])

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
torch.manual_seed(42)
train_dataloader = DataLoader(train_data,
                              batch_size=32,
                              shuffle=True)
test_dataloader = DataLoader(test_data,
                             batch_size=32,
                             shuffle=False)
len(train_dataloader), len(test_dataloader)

In [ ]:
train_feature_batch, train_label_batch = next(iter(train_dataloader))
train_feature_batch.shape, train_label_batch.shape
# DataLoader is not itterable thats why we using iter and next

In [ ]:
class FashionMNISTModelV0(nn.Module):
  def __init__(self, in_features, hidden_layers, output_features):
    super().__init__()
    self.layers_stacked = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=28*28, out_features=hidden_layers),
        nn.Linear(in_features=hidden_layers, out_features=len(class_names))
    )
  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.layers_stacked(x)

In [ ]:
model_0 = FashionMNISTModelV0(in_features=28*28, hidden_layers=10, output_features=len(class_names))
model_0.state_dict()

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

In [ ]:
torch.manual_seed(42)
from tqdm.auto import tqdm
epochs = 3
for epoch in tqdm(range(epochs)):
  train_loss = 0
  for batch, (X, y) in enumerate(train_dataloader):
    model_0.train()
    y_preds = model_0(X)
    loss = loss_fn(y_preds, y)
    train_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if batch % 400 == 0:
      print(f"Batch: {batch}/{train_dataloader}")
  train_loss /= len(train_dataloader)
  test_loss = 0
  model_0.eval()
  with torch.inference_mode():
    for X, y in test_dataloader:
      test_pred = model_0(X)
      loss = loss_fn(test_pred, y)
      test_loss += loss.item()
  test_loss /= len(test_dataloader)
  print(f"Epoch: {epoch + 1}, Train Loss: {train_loss}, Test Loss: {test_loss}")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
class FashinMNISTModel2(nn.Module):
  def __init__(self, input_features, hidden_layers, output_features):
    super().__init__()
    self.layers_stacked = nn.Sequential(
        nn.Flatten(),
        nn.Linear(input_features, hidden_layers),
        nn.ReLU(),
        nn.Linear(hidden_layers, output_features),
    )
  def forward(self, x):
    return self.layers_stacked(x)
model_1 = FashinMNISTModel2(input_features=28*28, hidden_layers=10, output_features=len(class_names))
next(iter(model_1.parameters()))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.1)

In [ ]:
import requests
from pathlib import Path

# Download helper functions from Learn PyTorch repo (if not already downloaded)
if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download")
else:
  print("Downloading helper_functions.py")
  # Note: you need the "raw" GitHub URL for this to work
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)
from helper_functions import accuracy_fn

In [ ]:
def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device = device):
  train_loss, train_acc = 0, 0
  model.to(device)
  for batch, (X, y) in enumerate(data_loader):
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    train_loss += loss.item()
    train_acc += accuracy_fn(y, y_pred.argmax(dim=1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  train_loss /= len(data_loader)
  train_acc /= len(data_loader)
  print(f"Train Loss: {train_loss}, Train Accuracy: {train_acc}")


In [ ]:
def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device = device):
  test_loss, test_acc = 0, 0
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)
      y_pred = model(X)
      test_loss += loss_fn(y_pred, y).item()
      test_acc += accuracy_fn(y, y_pred.argmax(dim=1))
  test_loss /= len(data_loader)
  test_acc /= len(data_loader)
  print(f"Test Loss: {test_loss}, Test Accuracy: {test_acc}")


In [ ]:
torch.manual_seed(42)
epochs = 3
for epoch in tqdm(range(epochs)):
  train_step(model=model_1,
             data_loader=train_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             accuracy_fn=accuracy_fn,
             device=device)
  test_step(model=model_1,
            data_loader=test_dataloader,
            loss_fn=loss_fn,
            accuracy_fn=accuracy_fn,
            device=device)

In [ ]:
torch.manual_seed(42)
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device = device):
    """Returns a dictionary containing the results of model predicting on data_loader.

    Args:
        model (torch.nn.Module): A PyTorch model capable of making predictions on data_loader.
        data_loader (torch.utils.data.DataLoader): The target dataset to predict on.
        loss_fn (torch.nn.Module): The loss function of model.
        accuracy_fn: An accuracy function to compare the models predictions to the truth labels.

    Returns:
        (dict): Results of model making predictions on data_loader.
    """
    loss, acc = 0, 0
    model.to(device)
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)
            # Make predictions with the model
            y_pred = model(X)

            # Accumulate the loss and accuracy values per batch
            loss += loss_fn(y_pred, y)
            acc += accuracy_fn(y_true=y,
                                y_pred=y_pred.argmax(dim=1)) # For accuracy, need the prediction labels (logits -> pred_prob -> pred_labels)

        # Scale loss and acc to find the average loss/acc per batch
        loss /= len(data_loader)
        acc /= len(data_loader)

    return {"model_name": model.__class__.__name__, # only works when model was created with a class
            "model_loss": loss.item(),
            "model_acc": acc}

# Calculate model 0 results on test dataset
model_0_results = eval_model(model=model_0, data_loader=test_dataloader,
    loss_fn=loss_fn, accuracy_fn=accuracy_fn
)
model_0_results

In [ ]:
model_1_results = eval_model(model=model_1,
                             data_loader=test_dataloader,
                             loss_fn=loss_fn,
                             accuracy_fn=accuracy_fn,
                             device=device)
model_1_results

In [ ]:
import torch
images = torch.randn(32, 3, 64, 64)
sample_image = images[0]
images.shape, sample_image.shape

In [ ]:
sample_image

In [ ]:
from torch import nn

In [ ]:
conv2d_layer = nn.Conv2d(in_channels=3,
                         out_channels=10,
                         kernel_size=3,
                         padding=0,
                         stride=1)
image = conv2d_layer(sample_image)

In [ ]:
image.shape

In [ ]:
conv2d_layer.weight.shape

In [ ]:
conv2d_layer

In [ ]:
conv2d_layer.bias.shape

In [ ]:
from torch.nn.modules.linear import Linear
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.conv import Conv2d
class FashionMNISTModelV3(nn.Module):
  def __init__(self, input_features, hidden_layers, output_features):
    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=hidden_layers, kernel_size=(3, 3), stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=(3, 3), stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layers*7*7, out_features=len(class_names)),
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    # print(x.shape)
    return x

model_3 = FashionMNISTModelV3(input_features=28*28, hidden_layers=10, output_features=len(class_names))


In [ ]:
next(iter(model_3.parameters()))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_3.parameters(), lr=0.1)

In [ ]:
epochs = 3
for epoch in tqdm(range(epochs)):
  train_step(model = model_3,
             data_loader = train_dataloader,
             loss_fn = loss_fn,
             optimizer = optimizer,
             accuracy_fn = accuracy_fn,
             device = device)
  test_step(model = model_3,
            data_loader = train_dataloader,
            loss_fn = loss_fn,
            accuracy_fn = accuracy_fn,
            device = device)


In [ ]:
image, label = next(iter(train_dataloader))

In [ ]:
image.shape

In [ ]:
class FashionMNISTModelV4(nn.Module):
  def __init__(self, input_features, hidden_layers, output_features):
    super().__init__()

    self.conv_block_1 = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=hidden_layers, kernel_size=(3, 3), stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )
    self.conv_block_2 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=(3, 3), stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
    )
    self.conv_block_3 = nn.Sequential(
        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=(3, 3), stride=1, padding=1),
        nn.ReLU(),

        nn.Conv2d(in_channels=hidden_layers, out_channels=hidden_layers, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=1),
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_layers*7*7, out_features=len(class_names)),
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = self.conv_block_1(x)
    # print(x.shape)
    x = self.conv_block_2(x)
    # print(x.shape)
    x = self.classifier(x)
    # print(x.shape)
    return x
model_4 = FashionMNISTModelV4(input_features=1, hidden_layers=10, output_features=len(class_names))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_4.parameters(), lr=0.1)

In [ ]:
# epochs = 10
# for epoch in tqdm(range(epochs)):
#   train_step(model = model_4,
#              data_loader = train_dataloader,
#              loss_fn = loss_fn,
#              optimizer = optimizer,
#              accuracy_fn = accuracy_fn,
#              device = device)
#   test_step(model = model_4,
#             data_loader = train_dataloader,
#             loss_fn = loss_fn,
#             accuracy_fn = accuracy_fn,
#             device = device)

In [ ]:
model_3_results = eval_model(model = model_3,
                             data_loader = test_dataloader,
                             loss_fn = loss_fn,
                             accuracy_fn = accuracy_fn,
                             device = device)
model_3_results

In [ ]:
import pandas as pd
results = pd.DataFrame({
    "Model 0": model_0_results,
    "Model 1": model_1_results,
    "Model 3": model_3_results,
})
results

In [ ]:
image, label = train_data[0]
image.shape, label

In [ ]:
def make_predictions(model: torch.nn.Module, device, data:list):
  pred_probs = []
  model.to(device)
  model.eval()
  with torch.inference_mode():
    for sample in data:
      print(f"Before unsqueeze: {sample.shape}")
      sample = torch.unsqueeze(sample, dim=0).to(device)
      print(f"After unsqueeze: {sample.shape}")
      pred_logit = model(sample)
      pred_prob = torch.softmax(pred_logit.squeeze(), dim=0)
      print(f"Pred Probabilities Shape: {pred_prob}")
      pred_probs.append(pred_prob.cpu())
  return torch.stack(pred_probs)

In [ ]:
import random
random.seed(42)
test_samples = []
test_labels = []
for sample, label in random.sample(list(test_data), k=9):
  sample = sample.to(device)
  test_samples.append(sample)
  test_labels.append(label)

In [ ]:
pred_probs = make_predictions(model=model_3,
                              device=device,
                              data=test_samples)
pred_probs[:2]

In [ ]:
pred_classes = pred_probs.argmax(dim=0)
pred_classes

In [ ]:
y_pred = []
model_3.eval()
with torch.inference_mode():
  for X, y in test_dataloader:
    X = X.to(device)
    y = y.to(device)
    pred_logits = model_3(X)
    # Model vraca shape: 32, 1, 10
    pred_probs = torch.softmax(pred_logits.squeeze(), dim=0)
    # Squeeze izbacuje dimenzije 1 i ostaje 32, 10
    pred_labels = pred_probs.argmax(dim=1)
    # Idemo kroz 1 dimenziju (10) i gledamo najvecu verovatnocu
    y_pred.append(pred_labels)
# y_pred[:10]
y_pred_tensor = torch.cat(y_pred)
print(f"Y predictions: {y_pred[:10]}\nY pred Tensor: {y_pred_tensor.shape}")

In [ ]:
!pip uninstall -y torchmetrics mlxtend
!pip install torchmetrics mlxtend

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix
confmat = ConfusionMatrix(num_classes=len(class_names), task="multiclass").to(device)
confmat_tesnor = confmat(preds=y_pred_tensor.to(device), target=test_data.targets.to(device))
fig, ax = plot_confusion_matrix(conf_mat=confmat_tesnor.cpu().numpy(), class_names=class_names, figsize=(10,7))

In [ ]:
from pathlib import Path

MODEL_PATH = Path('models')
MODEL_PATH.mkdir(parents=True,
                 exist_ok=True)
MODEL_NAME = 'pytorch_computer_vision_model_3'
print(f"Saving model to: {MODEL_PATH}")
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
torch.save(obj=model_3.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
loaded_model_3 = FashionMNISTModelV3(input_features=1,
                                     hidden_layers=10,
                                     output_features=len(class_names))
loaded_model_3.load_state_dict(torch.load(f=MODEL_SAVE_PATH))
loaded_model_3 = loaded_model_3.to(device)
loaded_model_3.state_dict()

In [ ]:
loaded_model_3_results = eval_model(model=loaded_model_3, data_loader=test_dataloader, loss_fn=loss_fn, accuracy_fn=accuracy_fn, device=device)
loaded_model_3_results

In [ ]:
model_3_results

In [ ]:
rand_tensor = torch.rand(size=(1, 3, 64, 64))
rand_tensor.shape

In [ ]:
tensor_in_conv2d = conv2d_layer(rand_tensor)
tensor_in_conv2d.shape

In [ ]:
conv2d_layer_v2 = Conv2d(in_channels=3, out_channels=20, kernel_size=3, padding=0, stride=2)
tensor_in_conv2d = conv2d_layer_v2(rand_tensor)
tensor_in_conv2d.shape


In [ ]:
y_preds = []
torch.manual_seed(42)
model_3.eval()
model_3.to(device)
with torch.inference_mode():
  for X, y in test_dataloader:
    X,y = X.to(device), y.to(device)
    y_logit = model_3(X)
    # print(y_logit.shape)
    y_pred_labels = torch.softmax(y_logit, dim=1).argmax(dim=1)
    y_preds.append(y_pred_labels)
fused_tensor = torch.cat(y_preds)
fused_tensor[:10]

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

confmat = ConfusionMatrix(num_classes=len(class_names), task=('multiclass')).to(device)
confmat_tensor = confmat(fused_tensor.to(device), test_data.targets.to(device))
fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.cpu().numpy(), figsize=(10,7), class_names=class_names)

In [ ]:
from pathlib import Path

MODEL_PATH = Path('exercise_model')
MODEL_PATH.mkdir(parents=True,
                 exist_ok=True)
MODEL_NAME = "03_pyroch_exercise_save_model_3"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME
print(f"Saving model {MODEL_NAME} to: {MODEL_SAVE_PATH}...")
torch.save(obj=model_3.state_dict(), f=MODEL_SAVE_PATH)

In [ ]:
loaded_model_4 = FashionMNISTModelV3(input_features=1, hidden_layers=10, output_features=len(class_names))
loaded_model_4.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
loaded_model_4_results = eval_model(model=loaded_model_4, data_loader=test_dataloader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn,
                                    device=device)
loaded_model_4_results

In [ ]:
model_3_results